In [ ]:
import numpy as np
import pandas as pd
from itertools import permutations
from K_meansClass import Kmeans

PATH = "D:\QG训练营\iris.data.txt"  #莺尾花数据集
Data_Iris = pd.read_csv(PATH)
print("Data_Iris即莺尾花数据集如下")
print(Data_Iris)

X_Iris = Data_Iris.iloc[:,0:4].values  #样本（4个特征）
print(f"样本数据X_Iris:{X_Iris}")
Y_Iris = Data_Iris.iloc[:,4].values  #莺尾花种类（3种）
print(f"真实标签Y_Iris:{Y_Iris}")

np.random.seed(2026)
kmeans = Kmeans(k=3, max_iter=100)
kmeans.fit(X_Iris)

print(f"各质心centroids：\n{kmeans.centroids:.4f}")

# ========== 获取聚类结果 ==========
predictions = kmeans.predict(X_Iris)
print(f"\n预测标签前20个: \n{predictions}")

# ========== 评估聚类效果 ==========
print("\n" + "=" * 50)
print("聚类结果分析")
print("=" * 50)


# 方法：对于每个簇，找出其中出现最多的真实类别
print("\n每个簇包含的样本数量及真实类别分布：")

# 将真实标签转换为数字（用于统计）
label_to_num = {
    'Iris-setosa': 0,
    'Iris-versicolor': 1,
    'Iris-virginica': 2
}
num_to_label = {0: 'setosa', 1: 'versicolor', 2: 'virginica'}

# 转换真实标签
Y_numeric = np.array([label_to_num[y] for y in Y_Iris])

# 统计每个簇的分布
cluster_distribution = {}
for cluster_id in range(3):
    mask = predictions == cluster_id
    cluster_size = np.sum(mask)

    if cluster_size > 0:
        # 统计该簇中各类别的数量
        unique, counts = np.unique(Y_numeric[mask], return_counts=True)
        cluster_distribution[cluster_id] = dict(zip(unique, counts))

        print(f"\n簇 {cluster_id} (共 {cluster_size} 个样本):")
        for label_num, count in zip(unique, counts):
            print(f"  - {num_to_label[label_num]}: {count} 个")

        # 找出该簇的主要类别
        main_label = unique[np.argmax(counts)]
        print(f"  → 主要类别: {num_to_label[main_label]}")
    else:
        print(f"\n簇 {cluster_id}: 空簇")

# ========== 计算聚类准确率（需要匹配簇编号和真实类别） ==========
print("\n" + "=" * 50)
print("聚类准确率计算")
print("=" * 50)

# 由于簇编号是任意的，我们需要找到最佳的编号映射
# 方法：尝试所有可能的映射，取准确率最高的
# 生成所有可能的映射（3! = 6种）
best_accuracy = 0
best_mapping = None

for mapping in permutations([0, 1, 2]):
    # mapping 是一个元组，如 (0,1,2) 表示簇0对应真实类别0，簇1对应真实类别1，簇2对应真实类别2
    # 将预测标签通过映射转换为真实标签格式
    mapped_predictions = np.array([mapping[p] for p in predictions])
    accuracy = np.mean(mapped_predictions == Y_numeric)

    if accuracy > best_accuracy:
        best_accuracy = accuracy
        best_mapping = mapping

print(
    f"最佳映射: 簇0→{num_to_label[best_mapping[0]]}, 簇1→{num_to_label[best_mapping[1]]}, 簇2→{num_to_label[best_mapping[2]]}")
print(f"聚类准确率: {best_accuracy:.4f} ({best_accuracy * 100:.2f}%)")

# 应用最佳映射，得到最终预测标签
final_predictions = np.array([best_mapping[p] for p in predictions])
